文档加载器 Document Loaders

数据源可能包含多种格式的文件，如文本文档、Markdown，PDF 等。LangChain 实现和集成了众多文档加载器（https://docs.langchain.com/oss/python/integrations/document_loaders ），方便从不同格式的文件中加载数据。

常用Loaders：
- TextLoader：文本文件。
- CSVLoader：CSV文件。
- PyPDFLoader：PDF文件。
- WebBaseLoader：网页。

LangChain的设计：对于 Source 中多种不同的数据源，我们可以用一种统一的形式读取、调用。上述每一个文档加载器，都要继承自 `BaseLoader` 基类，此类提供了通用的 load （一次加载所有文档）与 lazy_load （以延迟方式加载文档） 方法，用于从数据源加载数据并处理为 Document 对象 。

## 1、加载txt

In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader(file_path="../asset/load/01-langchain-utf-8.txt", encoding="utf-8")

docs = text_loader.load()

print(docs)

[Document(metadata={'source': '../asset/load/01-langchain-utf-8.txt'}, page_content='LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务')]


Document对象中有两个重要的属性：
- page_content：真正的文档内容，字符串类型。
- metadata：文档内容的原数据，字典类型。

In [3]:
print(type(docs[0]))
print(docs[0].metadata)
print(docs[0].page_content)

<class 'langchain_core.documents.base.Document'>
{'source': '../asset/load/01-langchain-utf-8.txt'}
LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务


## 2、加载CSV

举例：加载CSV所有列

In [6]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path="../asset/load/02-load.csv")
data = loader.load()
print(data)

print(type(data))
print(type(data[0]))
print(len(data))
print(data[0].metadata)
print(data[0].page_content)

[Document(metadata={'source': '../asset/load/02-load.csv', 'row': 0}, page_content='id: 1\ntitle: Introduction to Python\ncontent: Python is a popular programming language.\nauthor: John Doe'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 1}, page_content='id: 2\ntitle: Data Science Basics\ncontent: Data science involves statistics and machine learning.\nauthor: Jane Smith'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 2}, page_content='id: 3\ntitle: Web Development\ncontent: HTML, CSS and JavaScript are core web technologies.\nauthor: Mike Johnson'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 3}, page_content='id: 4\ntitle: Artificial Intelligence\ncontent: AI is transforming many industries.\nauthor: Sarah Williams')]
<class 'list'>
<class 'langchain_core.documents.base.Document'>
4
{'source': '../asset/load/02-load.csv', 'row': 0}
id: 1
title: Introduction to Python
content: Python is a popular programming language.
auth

## 3、加载JSON

LangChain提供的JSON格式的文档加载器是JSONLoader。在实际应用场景中，JSON格式的数据占很大比例，而且JSON的形式也是多样的，需要特别关注。

JSONLoader使用指定的jq结构来解析JSON文件。jq是一个轻量级的命令行JSON处理器，可以对JSON格式的数据进行各种复杂的处理，包括数据过滤、映射、减少和转换，是处理JSON数据的首选工具之一。
```
pip install jq
```
常见 jq schema 参考：
```
JSON -> ["...", "...", "..."]
jq_schema -> ".[]"

JSON -> [{"text": ...}, {"text": ...}, {"text": ...}]
jq_schema -> ".[].text"

JSON -> {"key": [{"text": ...}, {"text": ...}, {"text": ...}]}
jq_schema -> ".key[].text"
```
详细用法可参考 https://jqlang.org/manual/#basic-filters。

举例1：使用JSONLoader文档加载器加载

In [9]:
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# json_loader = JSONLoader(
#     file_path="../asset/load/03-load.json",
#     jq_schema=".", # 提取所有字段
#     text_content=False # 保持原始 JSON 结构，将提取的数据转换为JSON字符串存入page_content字段中
# )

# 遍历指定元素
json_loader = JSONLoader(
    file_path="../asset/load/03-load.json",
    jq_schema=".messages[].content"  # 遍历.messages[]中所有元素 从每一个元素中提取.content字段
)

docs = json_loader.load()
rprint(docs)


[
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-load.json',
            'seq_num': 1
        },
        page_content='Hello, how are you today?'
    ),
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-load.json',
            'seq_num': 2
        },
        page_content="I'm doing well, thanks for asking!"
    ),
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-load.json',
            'seq_num': 3
        },
        page_content='Would you like to meet for lunch?'
    ),
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-load.json',
            'seq_num': 4
        },
        page_content='Sure, that sounds great!'
    )
]

举例2：提取03-response.json文件中指定的文本

In [10]:
# 1.导入相关依赖
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# 2.定义json文件的路径
file_path = '../asset/load/03-response.json'

# 3.定义JSONLoader对象
# 需求1：提取data.items中的数据
loader = JSONLoader(
    file_path=file_path, # 文件路径
    jq_schema=".data.items[]",
    text_content=False, # 提取内容是否为字符串格式
)

# 需求2：提取data.items[].content中的数据
loader = JSONLoader(
    file_path=file_path, # 文件路径
    jq_schema=".data.items[].content",
)

loader = JSONLoader(
    file_path=file_path,
    jq_schema="""
        .data.items[] | {
            author,
            created_at,
            content: (.title + "\n" + .content)
        }
    """,
    text_content=False
)

data = loader.load()
rprint(data)

[
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-response.json',
            'seq_num': 1
        },
        page_content='{"author": {"id": "user_1", "name": "Alice"}, "created_at": "2023-10-05T08:12:33Z", 
"content": "Understanding JSONLoader\\nThis article explains how to parse API responses..."}'
    ),
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-response.json',
            'seq_num': 2
        },
        page_content='{"author": {"id": "user_2", "name": "Bob"}, "created_at": "2023-10-05T09:15:21Z", "content": 
"Advanced jq Schema Patterns\\nLearn to handle nested structures with..."}'
    ),
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-response.json',
            'seq_num': 3
        },
        page_content='{"author": {"id": "user_3", "name": "Charlie"}, "created_at": "2023-10-05T10:03:47Z", 
"content": "LangChain Metadata Handling\\nBest practices for preserving metadata..."}'
    )
]

## 4、加载pdf

PDF 存在多种来源格式，包括扫描版（图片 PDF）、电子文本版、混合版。并且布局格式也多种多样，包括单列布局、双列布局甚至竖排文本布局。并且包含段落、标题、页眉页脚、表格、数学公式、化学式、特殊符号、图片等各种元素。

因此，PDF 解析存在很多挑战。对于复杂 PDF，需要进行文本提取、布局检测、表格解析、公式识别等处理。

### 4.1 方式1:PyPDFLoader

LangChain加载PDF文件使用的是pypdf。
```
pip install pypdf
```


In [12]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    # 文件路径，支持本地文件和在线文件链接
    #file_path="../asset/load/04-sample.pdf",
    file_path="https://arxiv.org/pdf/alg-geom/9202012",
    # 提取模式:控制如何从 PDF 文件中解析和提取文本结构。
    # plain 提取文本，默认值
    # layout 布局感知提取模式，通常会通过插入大量的空格、换行符，来模拟原文档中的多栏、缩进和间距（适用场景：学术论文（如 arXiv 论文）、多栏报刊杂志、带有左右分栏的合同）
    extraction_mode="plain",
)

docs = loader.load()
print(len(docs))

2


### 4.2 方式2:MinerU

[MinerU](https://mineru.net/) 提供了 PDF、Word、PPT、图片等文件的解析，支持图像提取、OCR、公式、表格解析等功能。

调用在线服务：https://mineru.net/apiManage/docs。 可以从本地批量上传文件进行解析，并接收解析结果。

In [ ]:
import time
import requests
import os
from dotenv import load_dotenv

load_dotenv(override=True)

def upload_files(file_paths: list[str]) -> str:
    token = os.getenv("MINERU_TOKEN")
    url = "https://mineru.net/api/v4/file-urls/batch"
    header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

    files_info = [
        {
            "name": os.path.basename(file_path),
            "is_ocr": True, # 是否启动 ocr 功能
            "data_id": f"file_{i}" # 解析对象对应的数据 ID
        } for i, file_path in enumerate(file_paths)
    ]
    data = {
        "enable_formula": True,  # 是否开启公式识别
        "enable_table": True,  # 是否开启表格识别
        "language": "ch",
        "files": files_info,
    }

    try:
        response = requests.post(url,headers=header,json=data)
        if response.status_code == 200:
            result = response.json()
            print('response success. result:{}'.format(result))
            if result["code"] == 0:
                batch_id = result["data"]["batch_id"]
                urls = result["data"]["file_urls"]
                print('batch_id:{},urls:{}'.format(batch_id, urls))
                for i in range(0, len(urls)):
                    with open(file_paths[i], 'rb') as f:
                        res_upload = requests.put(urls[i], data=f)
                        if res_upload.status_code == 200:
                            print(f"{urls[i]} upload success")
                        else:
                            print(f"{urls[i]} upload failed")
                            return None
                return batch_id
            else:
                print('apply upload url failed,reason:{}'.format(result["msg"]))
                return None
        else:
            print('response not success. status:{} ,result:{}'.format(response.status_code, response))
            return None
    except Exception as err:
        print(err)
        return None

def download_files(batch_id):
    if not batch_id:
        print("batch_id为空，跳过下载")
        return

    os.makedirs("parsed_files", exist_ok=True)
    token = os.getenv("MINERU_TOKEN")
    url = f"https://mineru.net/api/v4/extract-results/batch/{batch_id}"
    header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

    failed_files = set()
    done_files = set()

    while True:
        res = requests.get(url, headers=header)
        result_json = res.json()

        if res.status_code != 200 or result_json.get("code") != 0:
            print("get result failed:", result_json)
            break

        extract_results = result_json["data"]["extract_result"]

        for result in extract_results:
            data_id = result["data_id"]
            if result["state"] == "failed":
                failed_files.add(data_id)
            elif result["state"] == "done" and data_id not in done_files:
                done_files.add(data_id)
                full_zip_url = result["full_zip_url"]
                res_download = requests.get(full_zip_url, stream=True)

                with open(f"parsed_files/{result['file_name']}_{result['data_id']}.zip", "wb") as f:
                    for chunk in res_download.iter_content(chunk_size=1024):
                        if chunk:
                            f.write(chunk)

        if len(failed_files) + len(done_files) == len(extract_results):
            break

        time.sleep(5)

    for i in failed_files:
        print("failed:", i)
    for i in done_files:
        print("done:", i)

file_paths = ["../asset/load/04-sample.pdf"]
batch_id = upload_files(file_paths)
print(batch_id)

if batch_id:
    download_files(batch_id)

## 5、加载word

可使用UnstructuredWordDocumentLoader加载word文件，需要unstructured包。

In [ ]:
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from rich import print as rprint

loader = UnstructuredWordDocumentLoader(
    file_path="../asset/load/05-openGauss调研.docx",
    # 加载模式，single-返回单个Document对象，elements-按标题等元素切分文档
    mode="single"
)

docs = loader.load()
print(len(docs))
rprint(docs)

## 6、加载Markdown

可使用UnstructuredMarkdownLoader加载Markdown文件，需要unstructured包。

举例1:使用UnstructuredMarkdownLoader加载Markdown文件

In [17]:

from langchain_community.document_loaders import UnstructuredMarkdownLoader

loader = UnstructuredMarkdownLoader(
    file_path="../asset/load/06-load.md",
    mode="single",
    # 解析策略，fast-快速模式，以最快的速度提取文本，不进行复杂的版面分析。hi_res-高分辨率模式
    strategy="fast"
)

docs = loader.load()
print(len(docs))
rprint(docs)

1


[
    Document(
        metadata={'source': '../asset/load/06-load.md'},
        page_content='自然语言处理技术文档\n\n本文档用于测试UnstructuredMarkdownLoader的中文处理能力。\n\n第一章：
简介\n\n自然语言处理(NLP)是人工智能的重要分支，主要技术包括：\n\n文本分类\n\n命名实体识别\n\n机器翻译\n\n情感分析\n
\n问答系统\n\n第二章：关键技术\n\n2.1 
预训练模型\n\nBERT：双向Transformer编码器\n\nGPT：自回归语言模型\n\nT5：文本到文本转换框架\n\n2.2 
代码示例\n\n```python from transformers import pipeline\n\n创建文本分类管道\n\nclassifier = 
pipeline("text-classification", model="bert-base-chinese")\n\nresult = classifier("这家餐厅的服务很棒！") 
print(result)'
    )
]

举例2:精细切割文档，保留结构信息

将Markdown文档按语义元素（标题、段落、列表、表格等）拆分成多个独立的小文档（ Element 对象），而不是返回单个大文档。通过指定 mode="elements" 轻松保持这种分离。

In [ ]:

from langchain_community.document_loaders import UnstructuredMarkdownLoader

loader = UnstructuredMarkdownLoader(
    file_path="../asset/load/06-load.md",
    mode="elements",
    # 解析策略，fast-快速模式，以最快的速度提取文本，不进行复杂的版面分析。hi_res-高分辨率模式
    strategy="fast"
)

docs = loader.load()
print(len(docs))
for doc in docs:
    rprint(doc.page_content)

## 7、加载HTML

In [ ]:

from langchain_community.document_loaders import UnstructuredHTMLLoader

# strategy:
#   "fast" 解析加载html文件速度是比较快（但可能丢失部分结构或元数据）
#   "hi_res": (高分辨率解析) 解析精准（速度慢一些）
#   "ocr_only" 强制使用ocr提取文本，仅仅适用于图像（对HTML无效）
# mode ：one of `{'paged', 'elements', 'single'}
#   "elements" 按语义元素（标题、段落、列表、表格等）拆分成多个独立的小文档
loader = UnstructuredHTMLLoader(
    file_path="../asset/load/07-load.html",
    mode="elements",
    strategy="fast"
)

docs = loader.load()
print(len(docs))

for doc in docs:
    rprint(doc)

## 8、加载File Directory

除了上述的单个文件加载，也可以批量加载一个文件夹内的所有文件。

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PythonLoader

loader = DirectoryLoader(
    path = "../asset/load",
    glob="*.py", # 文件匹配过滤，使用标准的Unix路径通配符
    use_multithreading=True, # 是否启用多线程，并发读取
    show_progress=True, # 是否显示进度条
    loader_cls=PythonLoader   # 指定底层核心加载器
)

docs = loader.load()
print(len(docs))
for doc in docs:
    rprint(doc)

## 9、了解BaseLoader、Document类

一方面，LangChain在设计时，要保证Source中多种不同的数据源，在接下来的流程中可以用一种统一的形式读取、调用。

另一方面，为什么PDFLoader和TextLoader等Document Loader都使用load()去加载，且都使用.page_content和.metadata读取数据。

【解答】每一个在LangChain中集成的文档加载器，都要继承自BaseLoader，BaseLoader提供了一个名为load的公开方法，用于从配置的不同数据源加载数据，全部作为Document对象。实现逻辑如下：

**BaseLoader类分析**

BaseLoader类定义了如何从不同的数据源加载文档，每个基于不同数据源实现的loader，都需要继承BaseLoader 。Baseloader要求不多，对于任何具体实现的loader，最少都要实现 load方法。
```
class BaseLoader(ABC):  # noqa: B024
    """Interface for document loader.

    Implementations should implement the lazy-loading method using generators to avoid
    loading all documents into memory at once.

    `load` is provided just for user convenience and should not be overridden.
    """

    # Sub-classes should not implement this method directly. Instead, they
    # should implement the lazy load method.
    def load(self) -> list[Document]:
        """Load data into `Document` objects.

        Returns:
            The documents.
        """
        return list(self.lazy_load())

    async def aload(self) -> list[Document]:
        """Load data into `Document` objects.

        Returns:
            The documents.
        """
        return [document async for document in self.alazy_load()]

    def load_and_split(
        self, text_splitter: TextSplitter | None = None
    ) -> list[Document]:
        """Load `Document` and split into chunks. Chunks are returned as `Document`.

        !!! danger

            Do not override this method. It should be considered to be deprecated!

        Args:
            text_splitter: `TextSplitter` instance to use for splitting documents.

                Defaults to `RecursiveCharacterTextSplitter`.

        Raises:
            ImportError: If `langchain-text-splitters` is not installed and no
                `text_splitter` is provided.

        Returns:
            List of `Document` objects.
        """
                if text_splitter is None:
            if not _HAS_TEXT_SPLITTERS:
                msg = (
                    "Unable to import from langchain_text_splitters. Please specify "
                    "text_splitter or install langchain_text_splitters with "
                    "`pip install -U langchain-text-splitters`."
                )
                raise ImportError(msg)

            text_splitter_: TextSplitter = RecursiveCharacterTextSplitter()
        else:
            text_splitter_ = text_splitter
        docs = self.load()
        return text_splitter_.split_documents(docs)
```
BaseLoader 把数据加载成 Documents object ，存到 Documents 类中的 page_content 中

**Document类分析**

Document 允许用户与文档的内容进行交互，可以查看文档内容。

其继承体系如下
```
Serializable
    ↑
BaseMedia
    ├── id
    ├── metadata
    ↑
Document
    ├── page_content
    ├── type = "Document"
```
Document源码
```
class Document(BaseMedia):
    """Class for storing a piece of text and associated metadata.

    !!! note

        `Document` is for **retrieval workflows**, not chat I/O. For sending text
        to an LLM in a conversation, use message types from `langchain.messages`.

    Example:
        ```python
        from langchain_core.documents import Document

        document = Document(
            page_content="Hello, world!", metadata={"source": "https://example.com"}
        )
        ```
    """

    page_content: str
    """String text."""

    type: Literal["Document"] = "Document"

    def __init__(self, page_content: str, **kwargs: Any) -> None:
        """Pass page_content in as positional or named arg."""
        # my-py is complaining that page_content is not defined on the base class.
        # Here, we're relying on pydantic base class to handle the validation.
        super().__init__(page_content=page_content, **kwargs)  # type: ignore[call-arg,unused-ignore]

    @classmethod
    def is_lc_serializable(cls) -> bool:
        """Return `True` as this class is serializable."""
        return True

    @classmethod
    def get_lc_namespace(cls) -> list[str]:
        """Get the namespace of the LangChain object.

        Returns:
            `["langchain", "schema", "document"]`
        """
        return ["langchain", "schema", "document"]

    def __str__(self) -> str:
        """Override `__str__` to restrict it to page_content and metadata.

        Returns:
            A string representation of the `Document`.
        """
        # The format matches pydantic format for __str__.
        #
        # The purpose of this change is to make sure that user code that feeds
        # Document objects directly into prompts remains unchanged due to the addition
        # of the id field (or any other fields in the future).
        #
        # This override will likely be removed in the future in favor of a more general
        # solution of formatting content directly inside the prompts.
        if self.metadata:
            return f"page_content='{self.page_content}' metadata={self.metadata}"
        return f"page_content='{self.page_content}'"
```
BaseMedia源码
```
class BaseMedia(Serializable):
    """Base class for content used in retrieval and data processing workflows.

    Provides common fields for content that needs to be stored, indexed, or searched.

    !!! note

        For multimodal content in **chat messages** (images, audio sent to/from LLMs),
        use `langchain.messages` content blocks instead.
    """

    # The ID field is optional at the moment.
    # It will likely become required in a future major release after
    # it has been adopted by enough VectorStore implementations.
    id: str | None = Field(default=None, coerce_numbers_to_str=True)
    """An optional identifier for the document.

    Ideally this should be unique across the document collection and formatted
    as a UUID, but this will not be enforced.
    """

    metadata: dict = Field(default_factory=dict)
    """Arbitrary metadata associated with the content."""
```